# Candidate new bicycle HIN corridors — victim map

This is an exploratory 2024–Sept. 1, 2026 screen of CRIS crashes outside the City's official Bicycle High Injury Network. Click points on the map for crash, victim and police-recorded factor details.

In [ ]:
from pathlib import Path
import html, zipfile, requests
import pandas as pd
import geopandas as gpd
from sklearn.cluster import AgglomerativeClustering

ROOT = Path.cwd(); RAW = ROOT/'data'/'raw'; BOUNDARIES = ROOT/'data'/'boundaries'; OUT = ROOT/'outputs'
BOUNDARIES.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

raw = pd.read_csv(RAW/'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY','A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type']=='3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity']=='K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity']=='A - SUSPECTED SERIOUS INJURY').astype(int)
factor_cols = ['Contributing Factor 1','Contributing Factor 2','Contributing Factor 3']
for col in factor_cols:
    if col not in target: target[col] = ''
target['contributing_factors'] = target[factor_cols].fillna('').astype(str).replace({'nan':'','None':''}, regex=False).agg('; '.join, axis=1).str.replace(r'(; )+','; ',regex=True).str.strip('; ')

def combine_values(s):
    return '; '.join(sorted({str(v).strip() for v in s.dropna() if str(v).strip() and str(v).strip().upper() not in {'NAN','NONE','TBD'}}))

sa = target[(target['City']=='SAN ANTONIO') & target['year'].between(2024,2026)].copy()
crashes = sa.groupby('Crash ID', as_index=False).agg(year=('year','first'), latitude=('latitude','first'), longitude=('longitude','first'), deaths=('death','sum'), serious_injuries=('serious_injury','sum'), victim_ages=('Person Age',combine_values), victim_genders=('Person Gender',combine_values), victim_helmets=('Person Helmet',combine_values), contributing_factors=('contributing_factors',combine_values))
points = gpd.GeoDataFrame(crashes.dropna(subset=['latitude','longitude']), geometry=gpd.points_from_xy(crashes.dropna(subset=['latitude','longitude'])['longitude'], crashes.dropna(subset=['latitude','longitude'])['latitude']), crs=4326)

hin_url = 'https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/SS4A_HIN_Dashboard_Data/FeatureServer/1/query?where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson'
r = requests.get(hin_url, timeout=120); r.raise_for_status()
hin_file = BOUNDARIES/'bicycle_hin_corridors.geojson'; hin_file.write_bytes(r.content)
hin = gpd.read_file(hin_file).to_crs(4326).rename(columns={'Name':'corridor'})
hin_buffer = hin.to_crs(2278); hin_buffer['geometry'] = hin_buffer.geometry.buffer(150)
inside = gpd.sjoin(points.to_crs(2278), hin_buffer[['bicycle_hin_id','geometry']], how='inner', predicate='within')
official_ids = set(inside['Crash ID']); outside_points = points[~points['Crash ID'].isin(official_ids)].copy()
print('Current qualifying crashes:', len(points)); print('Current crashes inside official HIN:', len(official_ids)); print('Current crashes outside official HIN:', len(outside_points))

In [ ]:
# Match crashes to local street segments and form bounded repeat-crash candidates.
streets_dir = RAW/'streets'; street_shp = streets_dir/'Streets'/'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(RAW/'Streets.zip') as z: z.extractall(streets_dir)
roads = gpd.read_file(street_shp).rename(columns={'CartID':'segmentid','MSAG_NAME':'road_label','FROM_STREE':'from_street','TO_STREET':'to_street','CoSARoadFu':'road_class'})
for col in ['segmentid','road_label','from_street','to_street','road_class']:
    if col not in roads: roads[col] = ''
if 'length_miles' not in roads: roads['length_miles'] = pd.to_numeric(roads.get('LengthFeet',0), errors='coerce')/5280
roads['length_miles'] = pd.to_numeric(roads['length_miles'], errors='coerce'); roads = roads[roads['length_miles'].fillna(0)>0].copy().to_crs(2279)
fields = ['segmentid','road_label','from_street','to_street','road_class','length_miles','geometry']
matches = gpd.sjoin_nearest(outside_points.to_crs(roads.crs), roads[fields], how='left', distance_col='match_distance_ft')
matches = matches[matches['match_distance_ft']<=150].sort_values('match_distance_ft').drop_duplicates('Crash ID').copy()
max_miles = float(pd.to_numeric(hin['Miles'], errors='coerce').max()); matches['candidate_group'] = pd.NA; group_number = 0
for road_name, group in matches.groupby('road_label', dropna=False):
    if len(group)<2 or pd.isna(road_name) or not str(road_name).strip(): continue
    coords = [[p.x,p.y] for p in group.geometry]; labels = AgglomerativeClustering(n_clusters=None, distance_threshold=max_miles*5280, linkage='complete').fit_predict(coords)
    for label in sorted(set(labels)):
        idx = group.index[labels==label]
        if len(idx)>=2: matches.loc[idx,'candidate_group'] = group_number; group_number += 1
matches = matches[matches['candidate_group'].notna()].copy()
def join_values(s): return ', '.join(sorted({str(v).strip() for v in s.dropna() if str(v).strip() and str(v).strip().upper() not in {'NAN','NONE','TBD'}}))
candidates = matches.groupby('candidate_group',as_index=False).agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'), first_year=('year','min'), last_year=('year','max'), roads=('road_label',join_values), from_streets=('from_street',join_values), to_streets=('to_street',join_values), max_match_distance_ft=('match_distance_ft','max'))
spans = []
for gid,g in matches.groupby('candidate_group'):
    spans.append({'candidate_group':gid,'span_miles':max(g.geometry.x.max()-g.geometry.x.min(),g.geometry.y.max()-g.geometry.y.min())/5280})
candidates = candidates.merge(pd.DataFrame(spans),on='candidate_group',how='left').sort_values(['crashes','deaths','serious_injuries'],ascending=False)
candidates.to_csv(OUT/'candidate_new_hin_corridors_2024_2026.csv',index=False); display(candidates)

In [ ]:
# Interactive map: orange candidate corridors; points are clickable.
import folium
segment_groups = matches[['segmentid','candidate_group']].dropna().drop_duplicates()
candidate_lines = roads.merge(segment_groups,on='segmentid',how='inner').dissolve(by='candidate_group',as_index=False).merge(candidates,on='candidate_group',how='left')
nearest = gpd.sjoin_nearest(points.to_crs(roads.crs), roads[['segmentid','road_label','geometry']], how='left', distance_col='match_distance_ft').sort_values('match_distance_ft').drop_duplicates('Crash ID')
point_info = points.merge(nearest[['Crash ID','road_label']],on='Crash ID',how='left')
def shown(v):
    s='' if pd.isna(v) else str(v).strip(); return html.escape(s) if s else 'Not recorded'
m = folium.Map(location=[point_info.geometry.y.mean(),point_info.geometry.x.mean()],zoom_start=11,tiles='OpenStreetMap',control_scale=True)
if not candidate_lines.empty:
    folium.GeoJson(candidate_lines.to_crs(4326).to_json(),name='Candidate corridors',style_function=lambda f:{'color':'#d95f02','weight':6,'opacity':.85},tooltip=folium.GeoJsonTooltip(fields=['roads','crashes','deaths','serious_injuries','first_year','last_year'],aliases=['Road','Crashes','Deaths','Serious injuries','First year','Last year'])).add_to(m)
for _,row in point_info.iterrows():
    outcome=f"{int(row['deaths'])} death(s), {int(row['serious_injuries'])} serious injury/ies"
    popup=(f"<b>Crash {shown(row['Crash ID'])}</b><br>Year: {int(row['year'])}<br>Road: {shown(row.get('road_label'))}<br>Outcome: {outcome}<br>"
           f"Victim age: {shown(row.get('victim_ages'))}<br>Victim gender: {shown(row.get('victim_genders'))}<br>Helmet: {shown(row.get('victim_helmets'))}<br>"
           f"Recorded contributing factors: {shown(row.get('contributing_factors'))}<br>Latitude: {row['latitude']:.6f}<br>Longitude: {row['longitude']:.6f}")
    folium.CircleMarker([row['latitude'],row['longitude']],radius=5,color='#e67e22' if row['deaths'] else '#3478a4',fill=True,fill_opacity=.9,tooltip=f"{int(row['year'])} — Crash {row['Crash ID']}",popup=folium.Popup(popup,max_width=360)).add_to(m)
folium.LayerControl().add_to(m); map_path=OUT/'candidate_corridors_map_2024_2026.html'; m.save(map_path); print('Map saved to:',map_path)